<img src="logos/logo_Facyt.png"
     width="250"
     style="display: block; margin-left: auto; margin-right: auto;">

# Pokemon - Data Preparation & Feature Engineering (Industrial Standard)

Este notebook construye una fase de preparacion de datos profesional para modelar batallas Pokemon.

Principios clave:
- Definir el target a nivel batalla (`first_wins`).
- Separar train/test antes de cualquier ajuste de pipeline.
- Evitar leakage (no usar `Winner`, `WinRate`, `Wins`, `n_combats`).
- Usar `Pipeline` y `ColumnTransformer` para transformaciones reproducibles.
- Persistir artefactos para el siguiente notebook de modelado.


## 1) Imports y configuracion


In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Alineada con EDA para mayor reproducibilidad transversal
RANDOM_STATE = 29
TEST_SIZE = 0.20
DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../artifacts")
TARGET_COL = "first_wins"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 2) Carga de datos
Se cargan las tablas base de Pokedex y combates.


In [14]:
pokemon_path = DATA_DIR / "pokemon.csv"
combats_path = DATA_DIR / "combats.csv"

pokemon_df = pd.read_csv(pokemon_path)
combats_df = pd.read_csv(combats_path)

print(f"pokemon_df shape: {pokemon_df.shape}")
print(f"combats_df shape: {combats_df.shape}")
display(pokemon_df.head(3))
display(combats_df.head(3))


pokemon_df shape: (800, 12)
combats_df shape: (50000, 3)


,#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False


,First_pokemon,Second_pokemon,Winner
0,266,298,298
1,702,701,701
2,191,668,668


## 3) Auditoria minima y reglas anti-leakage
Validamos estructura minima y declaramos variables prohibidas para entrenamiento.


In [15]:
required_pokemon_cols = {
    "#", "Name", "Type 1", "Type 2", "HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation", "Legendary"
}
required_combats_cols = {"First_pokemon", "Second_pokemon", "Winner"}

missing_pokemon_cols = sorted(required_pokemon_cols.difference(pokemon_df.columns))
missing_combats_cols = sorted(required_combats_cols.difference(combats_df.columns))

if missing_pokemon_cols:
    raise ValueError(f"Faltan columnas en pokemon_df: {missing_pokemon_cols}")
if missing_combats_cols:
    raise ValueError(f"Faltan columnas en combats_df: {missing_combats_cols}")

LEAKAGE_COLUMNS = {"Winner", "WinRate", "Wins", "n_combats"}
print("Columnas de leakage prohibidas como features:", sorted(LEAKAGE_COLUMNS))

pokemon_ids = set(pokemon_df["#"].astype(int))
combat_ids = set(combats_df["First_pokemon"]).union(set(combats_df["Second_pokemon"]))
coverage = len(combat_ids.intersection(pokemon_ids)) / len(combat_ids)
print(f"Cobertura de IDs de combate dentro de Pokedex: {coverage * 100:.2f}%")


Columnas de leakage prohibidas como features: ['WinRate', 'Winner', 'Wins', 'n_combats']
Cobertura de IDs de combate dentro de Pokedex: 100.00%


## 4) Limpieza estructural
Se eliminan duplicados exactos y se normalizan campos base.


In [16]:
initial_rows = len(combats_df)
combats_df = combats_df.drop_duplicates(subset=["First_pokemon", "Second_pokemon", "Winner"]).copy()
removed_duplicates = initial_rows - len(combats_df)

pokemon_df = pokemon_df.copy()
pokemon_df["Type 2"] = pokemon_df["Type 2"].fillna("None")
pokemon_df["is_mega"] = pokemon_df["Name"].str.contains(r"Mega|Primal", case=False, na=False).astype(int)
pokemon_df["Legendary"] = pokemon_df["Legendary"].astype(int)

print(f"Combates originales: {initial_rows:,}")
print(f"Combates tras deduplicar: {len(combats_df):,}")
print(f"Duplicados eliminados: {removed_duplicates:,}")


Combates originales: 50,000
Combates tras deduplicar: 48,048
Duplicados eliminados: 1,952


## 5) Funciones de feature engineering
Se construyen features a nivel batalla para representar ambos contendientes y sus diferencias.


In [ ]:
STATS_COLS = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]


def build_pokedex_lookup(pokemon_frame: pd.DataFrame) -> pd.DataFrame:
    """Crea tabla indexada por id para joins rapidos en combates."""
    lookup = pokemon_frame.copy()
    lookup["stats_total"] = lookup[STATS_COLS].sum(axis=1)
    return lookup.set_index("#")


def build_battle_level_dataset(combats_frame: pd.DataFrame, pokedex_lookup: pd.DataFrame) -> pd.DataFrame:
    """Construye dataset de modelado a nivel batalla con target y features de ambos Pokemon."""
    base = combats_frame[["First_pokemon", "Second_pokemon", "Winner"]].copy()
    base["first_wins"] = (base["Winner"] == base["First_pokemon"]).astype(int)

    # Clave de pareja no ordenada para split dependency-aware
    base["matchup_key"] = base.apply(
        lambda r: f"{min(r['First_pokemon'], r['Second_pokemon'])}_{max(r['First_pokemon'], r['Second_pokemon'])}",
        axis=1,
    )

    fields = STATS_COLS + ["stats_total", "Type 1", "Type 2", "Generation", "Legendary", "is_mega"]

    for prefix, id_col in [("first", "First_pokemon"), ("second", "Second_pokemon")]:
        mapped = pokedex_lookup.loc[:, fields].reindex(base[id_col]).reset_index(drop=True)
        mapped.columns = [f"{prefix}_{c}" for c in mapped.columns]
        base = pd.concat([base.reset_index(drop=True), mapped], axis=1)

    # Control de integridad posterior al join
    join_cols = [f"first_{c}" for c in fields] + [f"second_{c}" for c in fields]
    missing_join_rows = int(base[join_cols].isna().any(axis=1).sum())
    if missing_join_rows > 0:
        raise ValueError(
            f"Se detectaron {missing_join_rows} filas con joins incompletos. Revisar cobertura de IDs en Pokedex."
        )

    for col in STATS_COLS + ["stats_total"]:
        safe_col = col.lower().replace(". ", "_").replace(" ", "_")
        base[f"diff_{safe_col}"] = base[f"first_{col}"] - base[f"second_{col}"]
        base[f"abs_diff_{safe_col}"] = base[f"diff_{safe_col}"].abs()

    base["same_type1"] = (base["first_Type 1"] == base["second_Type 1"]).astype(int)
    base["same_type2"] = (base["first_Type 2"] == base["second_Type 2"]).astype(int)
    base["same_generation"] = (base["first_Generation"] == base["second_Generation"]).astype(int)
    base["both_legendary"] = ((base["first_Legendary"] == 1) & (base["second_Legendary"] == 1)).astype(int)

    return base


## 6) Construir dataset final de preparacion


In [18]:
pokedex_lookup = build_pokedex_lookup(pokemon_df)
battle_df = build_battle_level_dataset(combats_df, pokedex_lookup)

print(f"battle_df shape: {battle_df.shape}")
display(battle_df.head(3))


battle_df shape: (48048, 46)


,First_pokemon,Second_pokemon,Winner,first_wins,first_HP,first_Attack,first_Defense,first_Sp. Atk,first_Sp. Def,first_Speed,first_stats_total,first_Type 1,first_Type 2,first_Generation,first_Legendary,first_is_mega,second_HP,second_Attack,second_Defense,second_Sp. Atk,second_Sp. Def,second_Speed,second_stats_total,second_Type 1,second_Type 2,second_Generation,second_Legendary,second_is_mega,diff_hp,abs_diff_hp,diff_attack,abs_diff_attack,diff_defense,abs_diff_defense,diff_sp_atk,abs_diff_sp_atk,diff_sp_def,abs_diff_sp_def,diff_speed,abs_diff_speed,diff_stats_total,abs_diff_stats_total,same_type1,same_type2,same_generation,both_legendary
0,266,298,298,0,50,64,50,45,50,41,300,Rock,Ground,2,0,0,70,70,40,60,40,60,340,Grass,Dark,3,0,0,-20,20,-6,6,10,10,-15,15,10,10,-19,19,-40,40,0,0,0,0
1,702,701,701,0,91,90,72,90,129,108,580,Grass,Fighting,5,1,0,91,129,90,72,90,108,580,Rock,Fighting,5,1,0,0,0,-39,39,-18,18,18,18,39,39,0,0,0,0,0,1,1,1
2,191,668,668,0,55,40,85,80,105,40,405,Fairy,Flying,2,0,0,75,75,75,125,95,40,485,Psychic,None,5,0,0,-20,20,-35,35,10,10,-45,45,10,10,0,0,-80,80,0,0,0,0


## 7) Definicion de target y split dependency-aware
La separacion se hace antes de ajustar cualquier transformacion y evitando fuga por emparejamientos repetidos.


In [19]:
X = battle_df.drop(columns=[TARGET_COL])
y = battle_df[TARGET_COL].astype(int)

# Split por grupos de emparejamiento (pareja no ordenada) para reducir dependencia train-test
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=X["matchup_key"]))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Chequeo de no solapamiento de grupos entre train y test
train_groups = set(X_train["matchup_key"])
test_groups = set(X_test["matchup_key"])
group_overlap = len(train_groups.intersection(test_groups))

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Target train mean: {y_train.mean():.4f}")
print(f"Target test mean: {y_test.mean():.4f}")
print(f"Group overlap (debe ser 0): {group_overlap}")


X_train: (38438, 45), X_test: (9610, 45)
Target train mean: 0.4719
Target test mean: 0.4719


## 8) Feature selection y tipado
Se excluyen identificadores y columnas con leakage.


In [20]:
id_like_cols = ["First_pokemon", "Second_pokemon", "Winner", "matchup_key"]
leakage_cols = sorted(LEAKAGE_COLUMNS.intersection(X_train.columns))
drop_cols = [c for c in id_like_cols + leakage_cols if c in X_train.columns]

X_train_fe = X_train.drop(columns=drop_cols, errors="ignore").copy()
X_test_fe = X_test.drop(columns=drop_cols, errors="ignore").copy()

numeric_features = [
    c for c in X_train_fe.columns
    if c.startswith("diff_") or c.startswith("abs_diff_") or c in ["first_stats_total", "second_stats_total"]
]

categorical_features = [
    c for c in ["first_Type 1", "second_Type 1", "first_Type 2", "second_Type 2", "first_Generation", "second_Generation"]
    if c in X_train_fe.columns
]

binary_features = [
    c for c in ["first_Legendary", "second_Legendary", "first_is_mega", "second_is_mega", "same_type1", "same_type2", "same_generation", "both_legendary"]
    if c in X_train_fe.columns
]

print("Total features tras limpieza:", X_train_fe.shape[1])
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Binary:", len(binary_features))


Total features tras limpieza: 42
Numeric: 16
Categorical: 6
Binary: 8


## 9) Arquitectura de preprocessing
`ColumnTransformer` permite procesar cada tipo de feature con su estrategia correcta.


In [21]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop",
)

preprocess_pipeline


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

## 10) Validacion tecnica del pipeline
Se ajusta solo con train y se transforma train/test para verificar consistencia.


In [22]:
X_train_transformed = preprocess_pipeline.fit_transform(X_train_fe)
X_test_transformed = preprocess_pipeline.transform(X_test_fe)

print("Transformed train shape:", X_train_transformed.shape)
print("Transformed test shape:", X_test_transformed.shape)
print("Nulls en y_train:", int(y_train.isna().sum()))
print("Nulls en y_test:", int(y_test.isna().sum()))


Transformed train shape: (38438, 110)
Transformed test shape: (9610, 110)
Nulls en y_train: 0
Nulls en y_test: 0


## 11) Matrices finales para modelado
Se exponen objetos base para la siguiente fase de entrenamiento.


In [23]:
X_train_final = X_train_fe.copy()
X_test_final = X_test_fe.copy()

print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)


X_train_final: (38438, 42)
X_test_final: (9610, 42)


## 12) Persistencia de artefactos
Se guardan pipeline, split y manifiesto de features para reproducibilidad.


In [24]:
import joblib

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocess_pipeline, ARTIFACTS_DIR / "preprocess_pipeline_pokemon.joblib")
joblib.dump((X_train_final, X_test_final, y_train, y_test), ARTIFACTS_DIR / "split_data_pokemon.joblib")

feature_manifest = {
    "target": TARGET_COL,
    "drop_cols": drop_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "binary_features": binary_features,
    "notes": [
        "No se usan Winner/WinRate/Wins/n_combats como features por leakage.",
        "Se elimino duplicado exacto de combates antes de feature engineering.",
        "Type 2 se interpreta como ausencia estructural con valor None."
    ]
}

with open(ARTIFACTS_DIR / "feature_manifest_pokemon.json", "w", encoding="utf-8") as f:
    json.dump(feature_manifest, f, ensure_ascii=False, indent=2)

print("Artefactos guardados en:", ARTIFACTS_DIR.resolve())


Artefactos guardados en: /mnt/e/Documentos/Programas/pokemon-challenge-ml/notebooks/artifacts


## 13) Output summary
Este notebook produce:
- `X_train_final`, `X_test_final`, `y_train`, `y_test`
- `preprocess_pipeline` (fiteado sobre train)
- `../artifacts/preprocess_pipeline_pokemon.joblib`
- `../artifacts/split_data_pokemon.joblib`
- `../artifacts/feature_manifest_pokemon.json`

Ajustes metodologicos aplicados en esta version:
- Split dependency-aware por pareja de combate (`matchup_key`) para reducir fuga por dependencia.
- Verificacion explicita de no solapamiento de grupos entre train y test.
- Control de integridad post-join para detectar filas sin mapeo del Pokedex.
- Semilla alineada con EDA para consistencia de resultados entre fases.


## 14) Guia rapida de estudio (resumen)
Conceptos tecnicos que debes dominar para esta fase:
- Unidad de analisis: una fila = una batalla.
- Variable objetivo: `first_wins`.
- Leakage: cualquier variable derivada de resultados historicos globales.
- Feature engineering relacional: modelar diferencias entre oponentes (`diff_*`).
- Validacion dependency-aware: evitar que el mismo emparejamiento aparezca en train y test.
- Preprocessing por tipo: numericas, categoricas, binarias.
- Reproducibilidad: versionado, pipeline serializado y manifiesto de features.

Para estudio extendido, consulta el archivo `reports/guia_data_preparation_pokemon.md`.
